# 07 — Broadcast Dimension Join

Purpose: enrich Silver events with a small dimension table.

Process schema:

```text
SILVER EVENTS
+----------+---------+------------+--------+
| event_id | user_id | event_type | amount |
+----------+---------+------------+--------+
| e1       | u1      | purchase   | 10.0   |
| e2       | u2      | purchase   | 20.0   |
| e3       | u3      | refund     | 5.0    |
| e4       | u9      | purchase   | 50.0   |
+----------+---------+------------+--------+

SMALL DIM_USER
+---------+----------+---------+
| user_id | country  | segment |
+---------+----------+---------+
| u1      | SK       | retail  |
| u2      | CZ       | retail  |
| u3      | AT       | premium |
+---------+----------+---------+

                         BROADCAST JOIN on user_id
                                  |
                                  v

ENRICHED EVENTS
+----------+---------+------------+--------+----------+---------+
| event_id | user_id | event_type | amount | country  | segment |
+----------+---------+------------+--------+----------+---------+
| e1       | u1      | purchase   | 10.0   | SK       | retail  |
| e2       | u2      | purchase   | 20.0   | CZ       | retail  |
| e3       | u3      | refund     | 5.0    | AT       | premium |
+----------+---------+------------+--------+----------+---------+

UNKNOWN DIMENSION KEYS
+----------+---------+------------+--------+
| e4       | u9      | purchase   | 50.0   |
+----------+---------+------------+--------+
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("medallion-broadcast-dimension-join")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_medallion_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Silver events

Clean event data from Silver.

In [2]:
events_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("amount", DoubleType(), False),
])

event_rows = [
    ("e1", "u1", "purchase", 10.0),
    ("e2", "u2", "purchase", 20.0),
    ("e3", "u3", "refund", 5.0),
    ("e4", "u9", "purchase", 50.0),
]

silver_events = spark.createDataFrame(event_rows, events_schema)

silver_events.show(truncate=False)

+--------+-------+----------+------+
|event_id|user_id|event_type|amount|
+--------+-------+----------+------+
|e1      |u1     |purchase  |10.0  |
|e2      |u2     |purchase  |20.0  |
|e3      |u3     |refund    |5.0   |
|e4      |u9     |purchase  |50.0  |
+--------+-------+----------+------+



## Step 2 — Small dimension table

Small reference data is a good candidate for broadcast join.

In [3]:
dim_user_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("country", StringType(), False),
    StructField("segment", StringType(), False),
])

dim_rows = [
    ("u1", "SK", "retail"),
    ("u2", "CZ", "retail"),
    ("u3", "AT", "premium"),
]

dim_user = spark.createDataFrame(dim_rows, dim_user_schema)

dim_user.show(truncate=False)

+-------+-------+-------+
|user_id|country|segment|
+-------+-------+-------+
|u1     |SK     |retail |
|u2     |CZ     |retail |
|u3     |AT     |premium|
+-------+-------+-------+



## Step 3 — Broadcast join

Broadcast pattern:

```text
large/events table JOIN small/dimension table
```

Use:

```python
F.broadcast(dim_user)
```

In [4]:
enriched_events = (
    silver_events.alias("e")
    .join(
        F.broadcast(dim_user).alias("u"),
        on="user_id",
        how="inner"
    )
    .select(
        "event_id",
        "user_id",
        "event_type",
        "amount",
        "country",
        "segment"
    )
)

enriched_events.show(truncate=False)

+--------+-------+----------+------+-------+-------+
|event_id|user_id|event_type|amount|country|segment|
+--------+-------+----------+------+-------+-------+
|e1      |u1     |purchase  |10.0  |SK     |retail |
|e2      |u2     |purchase  |20.0  |CZ     |retail |
|e3      |u3     |refund    |5.0   |AT     |premium|
+--------+-------+----------+------+-------+-------+



## Step 4 — Unknown dimension keys

Inner join removes rows without a matching dimension key.

Keep missing keys visible:

```text
silver_events LEFT ANTI JOIN dim_user ON user_id
```

In [5]:
unknown_users = (
    silver_events.alias("e")
    .join(
        dim_user.select("user_id").alias("u"),
        on="user_id",
        how="left_anti"
    )
    .withColumn("error_reason", F.lit("missing_dim_user"))
)

unknown_users.show(truncate=False)

+-------+--------+----------+------+----------------+
|user_id|event_id|event_type|amount|error_reason    |
+-------+--------+----------+------+----------------+
|u9     |e4      |purchase  |50.0  |missing_dim_user|
+-------+--------+----------+------+----------------+



## Step 5 — Gold by dimension attributes

After enrichment, Gold can aggregate by dimension columns.

In [6]:
gold_by_country_segment = (
    enriched_events
    .groupBy("country", "segment")
    .agg(
        F.count("*").alias("event_count"),
        F.sum(F.when(F.col("event_type") == "purchase", F.col("amount")).otherwise(F.lit(0.0))).alias("gross_sales"),
        F.sum(F.when(F.col("event_type") == "refund", F.col("amount")).otherwise(F.lit(0.0))).alias("refund_amount"),
    )
    .withColumn("net_sales", F.col("gross_sales") - F.col("refund_amount"))
    .orderBy("country", "segment")
)

gold_by_country_segment.show(truncate=False)

+-------+-------+-----------+-----------+-------------+---------+
|country|segment|event_count|gross_sales|refund_amount|net_sales|
+-------+-------+-----------+-----------+-------------+---------+
|AT     |premium|1          |0.0        |5.0          |-5.0     |
|CZ     |retail |1          |20.0       |0.0          |20.0     |
|SK     |retail |1          |10.0       |0.0          |10.0     |
+-------+-------+-----------+-----------+-------------+---------+



## Step 6 — Explain physical plan

For a small dimension, Spark should use broadcast join.

In [7]:
enriched_events.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [event_id#0, user_id#1, event_type#2, amount#3, country#18, segment#19]
   +- BroadcastHashJoin [user_id#1], [user_id#17], Inner, BuildRight, false
      :- Scan ExistingRDD[event_id#0,user_id#1,event_type#2,amount#3]
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=353]
         +- Scan ExistingRDD[user_id#17,country#18,segment#19]




## Step 7 — Control totals

```text
input events = enriched events + unknown dimension keys
```

In [8]:
control_totals_schema = StructType([
    StructField("metric", StringType(), False),
    StructField("value", LongType(), False),
])

control_totals = spark.createDataFrame(
    [
        ("silver_event_rows", silver_events.count()),
        ("dim_user_rows", dim_user.count()),
        ("enriched_event_rows", enriched_events.count()),
        ("unknown_user_rows", unknown_users.count()),
    ],
    control_totals_schema
)

control_totals.show(truncate=False)

+-------------------+-----+
|metric             |value|
+-------------------+-----+
|silver_event_rows  |4    |
|dim_user_rows      |3    |
|enriched_event_rows|3    |
|unknown_user_rows  |1    |
+-------------------+-----+



## Final result

```text
SILVER EVENTS
      |
      v
BROADCAST JOIN DIM_USER
      |
      +--> ENRICHED EVENTS
      |
      +--> UNKNOWN DIMENSION KEYS
      |
      v
GOLD BY COUNTRY / SEGMENT
```

In [9]:
spark.stop()